In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install gymnasium sb3-contrib stable-baselines3 --quiet

In [ ]:
# ── 2. System path setup ─────────────────────────────────────────────────────
import sys, os
import numpy as np

DATASET_PATH = "/kaggle/input/datasets/nguyennhatphong/roguelike-rl-env"
sys.path.insert(0, DATASET_PATH)

print("Python version:", sys.version)
print("Dataset files:", os.listdir(DATASET_PATH)[:10])

In [ ]:
# ── 3. Aggressive Environment Wrapper ───────────────────────────────────────
import gymnasium as gym
from gymnasium import spaces
from python_roguelike.env.roguelike_env import RoguelikeEnv
from python_roguelike.data.enums import GameState


class AggressiveEnv(RoguelikeEnv):
    """
    Aggressive agent reward shaping:
    - Heavy time penalty (-1 per turn) to encourage fast kills
    - Scaled floor reward (+30 per floor cleared)
    - Big game-win bonus (+50) and game-loss penalty (-100)
    - Light HP penalty (-0.1 per HP lost) — accepted trade-off

    With ~500-700 turns to win the game, -1 per turn creates strong
    pressure. Splitting rewards per floor prevents the agent from
    gaming early floors then dying.
    """

    def __init__(self, seed=42, json_path=None, max_steps=10_000):
        super().__init__(seed=seed, json_path=json_path, max_steps=max_steps)
        self._prev_floor_custom = 0

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self._prev_floor_custom = self._controller.current_run.current_floor
        return obs, info

    def step(self, action: int):
        run = self._controller.current_run
        prev_hp = run.the_hero.current_health
        prev_floor = run.current_floor

        obs, base_reward, terminated, truncated, info = super().step(action)

        # ── Override/recalculate reward with aggressive shaping ──────────────
        run = self._controller.current_run
        hero = run.the_hero
        reward = 0.0

        # Time penalty: every turn costs -1
        reward -= 1.0

        # HP penalty: light punishment for taking damage
        hp_delta = hero.current_health - prev_hp
        if hp_delta < 0:
            reward += hp_delta * 0.1  # -x HP → -0.x reward

        # Floor progress bonus
        floor_delta = run.current_floor - prev_floor
        if floor_delta > 0:
            reward += floor_delta * 30.0  # +30 per floor cleared

        # Terminal rewards
        if terminated:
            if hero.current_health > 0:
                # Won the game — +50 on top of floor rewards already given
                reward += 50.0
            else:
                # Lost — heavy penalty
                reward -= 100.0

        return obs, reward, terminated, truncated, info


# Quick smoke test
json_path = os.path.join(DATASET_PATH, "python_roguelike", "GameData.json")
env = AggressiveEnv(seed=42, json_path=json_path)
obs, info = env.reset()
print("Env created — obs shape:", obs.shape)
print("Action space:", env.action_space)
print("Initial info:", info)

In [ ]:
# ── 4. Validate Action Masking ───────────────────────────────────────────────
# Run 100 random masked steps to confirm masking works correctly
from sb3_contrib.common.wrappers import ActionMasker

def mask_fn(env_inner):
    return env_inner.action_masks()

masked_env = ActionMasker(env, mask_fn)
obs, info = masked_env.reset()

illegal_actions = 0
for step in range(100):
    mask = masked_env.action_masks()
    valid_actions = np.where(mask)[0]
    action = np.random.choice(valid_actions)  # always pick valid
    obs, reward, terminated, truncated, info = masked_env.step(action)
    if terminated or truncated:
        obs, info = masked_env.reset()

print(f"   Masking validation: 0/{100} illegal actions taken")
print(f"   Final state: floor={info['floor']}, hp={info['hp']}/{info['max_hp']}")

In [ ]:
# ── 5. Vectorized Environment for Faster Training ────────────────────────────
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from sb3_contrib.common.wrappers import ActionMasker

N_ENVS = 4  # Parallel environments

def make_env(seed_offset: int):
    def _init():
        _env = AggressiveEnv(
            seed=1000 + seed_offset,
            json_path=json_path,
            max_steps=10_000,
        )
        _env = ActionMasker(_env, lambda e: e.action_masks())
        return _env
    return _init

vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])
vec_env = VecMonitor(vec_env)

print(f"   VecEnv ready: {N_ENVS} parallel envs")

In [ ]:
# ── 6. MaskablePPO Model ─────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO

policy_kwargs = dict(
    net_arch=dict(
        pi=[256, 256],
        vf=[256, 256],
    )
)

model = MaskablePPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    n_steps=2048, 
    batch_size=512,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.995,    
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.02, 
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    tensorboard_log="/kaggle/working/tensorboard/aggressive",
)

print("Model created. Policy params:", sum(p.numel() for p in model.policy.parameters()))

In [ ]:
# ── 7. Callbacks: Checkpoint + Evaluation ────────────────────────────────────
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
    CallbackList,
)
from sb3_contrib.common.wrappers import ActionMasker

# Evaluation env
eval_env = ActionMasker(
    AggressiveEnv(seed=9999, json_path=json_path),
    lambda e: e.action_masks()
)

checkpoint_cb = CheckpointCallback(
    save_freq=50_000,
    save_path="/kaggle/working/checkpoints/aggressive",
    name_prefix="aggressive_ppo",
    verbose=1,
)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path="/kaggle/working/best_model/aggressive",
    log_path="/kaggle/working/eval_logs/aggressive",
    eval_freq=25_000,
    n_eval_episodes=10,
    deterministic=True,
    verbose=1,
)

callbacks = CallbackList([checkpoint_cb, eval_cb])
print("Callbacks configured")

In [ ]:
# ── 8. Training ──────────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 1_000_000

print(f"   Starting Aggressive Agent training for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"   Strategy: Time penalty (-1/turn) + Floor rewards (+30/floor)")
print(f"   Win: +30/floor + +50 full clear | Lose: -100")
print()

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callbacks,
    progress_bar=True,
)

model.save("/kaggle/working/aggressive_ppo_final")
print("\n   Training complete! Model saved to: /kaggle/working/aggressive_ppo_final")

In [ ]:
# ── 9. Evaluation ────────────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO
import numpy as np

model = MaskablePPO.load("/kaggle/working/best_model/aggressive/best_model")

N_EVAL = 50
floors = []
wins = 0
turns_per_win = []

for ep in range(N_EVAL):
    eval_env_ep = AggressiveEnv(seed=ep, json_path=json_path)
    eval_env_ep = ActionMasker(eval_env_ep, lambda e: e.action_masks())
    obs, info = eval_env_ep.reset()
    done = False
    turns = 0

    while not done:
        action, _ = model.predict(obs, deterministic=True, action_masks=eval_env_ep.action_masks())
        obs, reward, terminated, truncated, info = eval_env_ep.step(int(action))
        turns += 1
        done = terminated or truncated

    floors.append(info["floor"])
    if info["hp"] > 0 and terminated:
        wins += 1
        turns_per_win.append(turns)

print(f"\n Aggressive Agent Evaluation Results ({N_EVAL} episodes)")
print(f"═" * 50)
print(f"  Win Rate:          {wins/N_EVAL*100:.1f}%  ({wins}/{N_EVAL})")
print(f"  Avg Floor Reached: {np.mean(floors):.1f} / 15")
print(f"  Max Floor Reached: {max(floors)}")
if turns_per_win:
    print(f"  Avg Turns per Win: {np.mean(turns_per_win):.0f}")
    print(f"  Min Turns per Win: {min(turns_per_win)}  (fewer = more aggressive!)")